# GCP Enterprise Multi-Agent Data Mesh Setup & Runbook

Welcome to the developer runbook for the **Enterprise Agentic Data Mesh**! 
This notebook provides step-by-step instructions to:
1. Set up the environment & authenticate with Google Cloud Platform.
2. Create and deploy the schemas for **BigQuery**, **Spanner**, **AlloyDB**, and **Dataplex**.
3. Ingest interconnected test data across all enterprise domains.
4. Trigger complex cross-domain autonomous AI queries.

### Architecture Highlights
This system leverages a **Master Data Orchestrator** using the **Agent Development Kit (ADK)** to coordinate specialized domain agents:
- 📊 **AnalyticsAgent**: Speaks to **BigQuery** (EDW) and **AlloyDB** (CRM).
- 🛒 **RetailAgent**: Speaks to **Cloud Spanner** (Global Retail & Supply Chain) leveraging **Spanner Graph**.
- 💼 **FinancialAgent**: Speaks to **Oracle DB@GCP** (ERP & Financial Ledgers).
- 👥 **HRAgent**: Speaks to the talent and recruitment pipeline data.
- 🛡️ **GovernanceAgent**: Enforces Zero-Trust cell-level masking and compliance check gates.

## Section 1: Installation & Environment Setup

First, let's install the required Node.js and system dependencies, and verify our active workspace configurations.

In [ ]:
# Install workspace dependencies
!npm install

# Check active environment configurations
!cat .env

## Section 2: GCP Authentication

To provision and interact with live GCP services, we must authenticate using the `gcloud` SDK. 
Run the cells below to authenticate your user account and activate Application Default Credentials (ADC).

In [ ]:
# Authenticate user account with GCP
!gcloud auth login --no-launch-browser

# Configure application default credentials for Node.js/Python client libraries
!gcloud auth application-default login --no-launch-browser

## Section 3: Deploying GCP Database Schemas

With GCP authenticated, we will now deploy the standard schema DDLs for each database in the mesh.

### 1. BigQuery (Enterprise Data Warehouse)
We will create the `marketing_edw` dataset and execute the analytical schemas containing customer segments and compliance logs.

In [ ]:
# Create BigQuery dataset
!bq mk --force=true marketing_edw

# Deploy BigQuery marketing EDW schemas
!bq query --use_legacy_sql=false < db-schemas/bigquery_schema.sql

### 2. Cloud Spanner (Global Retail & Inventory)
We will provision a regional Spanner instance (`global-retail-instance`) in `us-central1`, create a database (`global-retail-db`), and deploy the retail schema containing Spanner Graphs and transactions.

In [ ]:
# Create Spanner Instance (approx. 1-2 mins)
!gcloud spanner instances create global-retail-instance \
    --config=regional-us-central1 \
    --description="Global Retail Mesh" \
    --nodes=1

# Create Spanner Database
!gcloud spanner databases create global-retail-db --instance=global-retail-instance

# Apply Spanner retail and graph schemas
!gcloud spanner databases ddl update global-retail-db \
    --instance=global-retail-instance \
    --file=db-schemas/spanner_schema.sql

### 3. AlloyDB & Dataplex Setup
For AlloyDB and Dataplex, we instantiate high-performance databases and govern them using aspect schemas.

Run the Dataplex provisioning helper script to configure the data lakes and zones.

In [ ]:
# Deploy Dataplex resources
!node scripts/create_dataplex_resources.js

## Section 4: Ingesting Connected Test Data

To demonstrate the power of the multi-agent mesh, we generate interconnected test datasets that span across all domains:
- **BigQuery**: VIP customer segmentation records (`bigquery_segments.csv`).
- **AlloyDB**: Delayed shipment support tickets (`alloydb_tickets.csv`).
- **Spanner**: NYC Store inventory transactions (`spanner_transactions.csv`).
- **Oracle**: Delayed purchase orders (`oracle_orders.csv`).

Let's generate and inspect this connected test data.

In [ ]:
# Generate the interconnected test datasets
!node scripts/generate_test_data.js

# Preview BigQuery segment records
!head -n 5 test-data/bigquery_segments.csv

# Preview Spanner transaction records
!head -n 5 test-data/spanner_transactions.csv

# Preview Oracle ERP Purchase Order records
!head -n 5 test-data/oracle_orders.csv

## Section 5: Triggering Autonomous Agent Queries

Now we trigger the **Autonomous A2A (Agent-to-Agent) Data Mesh**! 
We use the non-interactive query runner (`scripts/query_runner.js`) to send complex queries to the Orchestrator and observe how it maps business terms, delegates sub-tasks to specialized sub-agents, reflects on the synthesis, and returns the final answers.

### Query 1: Cross-Domain Recruitment & Retail Impact
We ask the mesh to analyze how HR recruitment delays affect the Spanner supply chain for high-value VIP segments.

In [ ]:
!node scripts/query_runner.js "Analyze how recruitment delays in HR are affecting our Spanner Global supply chain for high-value customers identified in BigQuery risk segments."

### Query 2: Multi-Domain Inventory and PO Delays
We ask the mesh to correlate VIP customer transactions in Spanner with delayed PO shipments in Oracle ERP.

In [ ]:
!node scripts/query_runner.js "Show me standard VIP segment customer transactions from Spanner that have delayed PO shipments in Oracle ERP."

### Query 3: Zero-Trust Compliance Masking Gate
We trigger a security test query. Watch how the Governance Agent automatically masks employee salaries and sensitive PII data, returning `[MASKED]` fields to satisfy the EMEA privacy policy rules!

In [ ]:
!node scripts/query_runner.js "Provide a list of HR department employees, their email addresses, and their associated salaries."

## Section 6: Federated Data Mesh Governance

Enterprise data security requires a robust, federated governance system. In our Agentic Data Mesh, **zero-trust policy enforcement** is managed by the `GovernanceAgent`.

Policies are configured declaratively in `config/policies.json`. Let's examine the active policies that enforce cell-level masking and PII constraints.

In [ ]:
# Display active data governance policies and masking rules
!cat config/policies.json

## Section 7: Schema & Metadata Federation

Rather than copying data into a central repository, our data mesh relies on a **Federated Catalog**. 
The `CatalogAgent` and the `MetadataCatalog` dynamically map schemas and assets from disparate domains (Spanner inventory tables, BigQuery EDW logs, Oracle ERP POs) to let the orchestrator understand where information is located.

Let's inspect the data source catalog mapping that describes where schemas are hosted and how they link to their respective business domains.

In [ ]:
# Print the federated data source registry configuration
!cat config/data_sources.json

## Section 8: Data Contracts (Trust but Verify)

Every domain agent exposes its insights and data as a standardized **Data Product**. 
Before a sub-agent's data product can be consumed by the Master Orchestrator, it is automatically validated against a strict **Data Contract** (`config/data_contracts.json`) using the `validateDataProduct` gate.

Let's inspect the declarative Data Product Contracts defined for our mesh.

In [ ]:
# Print active Data Product Contracts in the mesh
!cat config/data_contracts.json

## Section 9: Cross-Domain Knowledge Graph (GraphRAG & Lineage)

To guarantee that sub-agents' logical leaps are grounded in actual database relationships, the mesh uses a unified **Knowledge Graph** (`agent/utils/kg_service.js`) and a **GraphRAG Grounding** engine (`agent/utils/grounding.js`).

Every user intent triggers:
1. A lineage intent node in the local Knowledge Graph.
2. Real-time GraphRAG traversal mapping dependencies (e.g., connecting a VIP customer to an NYC Store inventory purchase order, and back to a delayed logistics shipment PO).
3. Attaching formal citations that anchor sub-agent insights to verified graph paths.

Let's view the dynamic GraphRAG grounding implementation in the mesh.

In [ ]:
# Display the active GraphRAG grounding module
!cat agent/utils/grounding.js